# <font color="#418FDE" size="6.5" uppercase>**Output Structures**</font>

>Last update: 20260714.
    
By the end of this Lecture, you will be able to:
- Apply multiclass strategies for estimators that need decomposition wrappers. 
- Model multilabel and multioutput classification targets in scikit-learn. 
- Use chained estimators for dependent classification or regression outputs. 


## **1. Multiclass Strategies**

### **1.1. Native Multiclass Support**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_19/Lecture_A/image_01_01.jpg?v=1784013783" width="250">



>* Handles three or more classes directly
>* Avoids separate binary decomposition strategies

>* Train directly on all class labels
>* Avoid extra multiclass decomposition wrappers

>* Check estimator assumptions against messy class patterns
>* Evaluate each class, not just accuracy



In [ ]:
#@title Python Code - Native Multiclass Support

# Demonstrate native multiclass classification support.
# Fit one estimator directly to three classes.
# Compare predictions with true iris labels.

import sklearn
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Load a small packaged dataset with three flower classes.
iris = load_iris()
features = iris.data
target = iris.target

# Validate that this is truly a multiclass target.
class_count = len(iris.target_names)
if class_count < 3:
    raise ValueError("This example needs at least three classes.")

# Split data while preserving class proportions in both sets.
X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.3, stratify=target, random_state=42
)

# Fit one native multiclass tree without any decomposition wrapper.
model = DecisionTreeClassifier(max_depth=3, random_state=42)
model.fit(X_train, y_train)

# Predict one label from all three possible classes directly.
predicted = model.predict(X_test)
accuracy = accuracy_score(y_test, predicted)

# Show concise evidence of direct multiclass learning.
print("scikit-learn version:", sklearn.__version__)
print("Number of classes learned directly:", len(model.classes_))
print("Class names:", ", ".join(iris.target_names))
print("Test accuracy:", round(accuracy, 3))
print("First 8 true labels:", list(y_test[:8]))
print("First 8 predicted labels:", list(predicted[:8]))



### **1.2. One Versus Rest**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_19/Lecture_A/image_01_02.jpg?v=1784013785" width="250">



>* Train one binary classifier per class
>* Choose the class with strongest score

>* Simple scaling with one classifier per class
>* Binary models specialize in class-specific signals

>* Binary tasks can become class-imbalanced
>* Compare classifier scores carefully for final predictions



In [ ]:
#@title Python Code - One Versus Rest

# Demonstrate one versus rest multiclass classification.
# Wrap a binary logistic model for iris classes.
# Compare class scores for one test flower.

import sklearn
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Load a small built-in multiclass dataset.
iris = load_iris()
X = iris.data
y = iris.target

# Validate the expected multiclass target shape.
if len(set(y)) != 3:
    raise ValueError("This lesson expects exactly three iris classes.")

# Split data while preserving class proportions.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

# Wrap a binary classifier with one-versus-rest decomposition.
binary_logistic = LogisticRegression(max_iter=300, random_state=42)
model = make_pipeline(StandardScaler(), OneVsRestClassifier(binary_logistic))

# Fit once; the wrapper trains one binary model per class.
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

# Inspect comparable scores from the three binary classifiers.
scores = model.decision_function(X_test[:1])[0]
score_text = ", ".join(
    f"{name}: {score:.2f}" for name, score in zip(iris.target_names, scores)
)

print(f"scikit-learn version: {sklearn.__version__}")
print(f"Binary classifiers trained: {len(model.named_steps['onevsrestclassifier'].estimators_)}")
print(f"Test accuracy: {accuracy_score(y_test, y_pred):.2f}")
print(f"Scores for one test flower: {score_text}")
print(f"Predicted class: {iris.target_names[y_pred[0]]}")



### **1.3. One Versus One**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_19/Lecture_A/image_01_03.jpg?v=1784013787" width="250">



>* Train classifiers for every class pair
>* Pairwise votes choose the final class

>* Focused pairwise models capture subtle class differences
>* Many classes require many manageable classifiers

>* Wrapper builds and combines pairwise classifiers
>* Evaluate votes to catch hidden class confusions



In [ ]:
#@title Python Code - One Versus One

# Demonstrate one versus one multiclass classification.
# Pairwise classifiers vote for final classes.
# Compare wrapper size and test accuracy.

import sklearn
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.multiclass import OneVsOneClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Load a small built-in three-class dataset.
iris = load_iris()
features = iris.data
target = iris.target

# Validate the expected multiclass teaching setup.
if len(set(target)) != 3:
    raise ValueError("This example needs exactly three classes.")

# Split data while preserving class proportions.
features_train, features_test, target_train, target_test = train_test_split(
    features, target, test_size=0.3, stratify=target, random_state=42
)

# Wrap a binary-style classifier with one versus one.
base_model = LogisticRegression(max_iter=200, random_state=42)
ovo_model = OneVsOneClassifier(base_model)
ovo_model.fit(features_train, target_train)

# Each pair of classes gets one fitted binary classifier.
class_count = len(ovo_model.classes_)
pairwise_count = len(ovo_model.estimators_)
expected_count = class_count * (class_count - 1) // 2

# Evaluate the combined voting prediction on held-out data.
predicted = ovo_model.predict(features_test)
accuracy = accuracy_score(target_test, predicted)

print(f"scikit-learn version: {sklearn.__version__}")
print(f"Classes: {class_count}; pairwise classifiers: {pairwise_count}")
print(f"Expected one-vs-one classifiers: {expected_count}")
print(f"Test accuracy from pairwise voting: {accuracy:.2f}")
print(f"First five predicted class names: {list(iris.target_names[predicted[:5]])}")



## **2. Multilabel Outputs**

### **2.1. Multilabel Classification**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_19/Lecture_A/image_02_01.jpg?v=1784013791" width="250">



>* Samples can have multiple labels simultaneously
>* Targets use rows and label columns

>* Split labels into separate binary tasks
>* Combine predictions, but label independence is simplified

>* Measure partial correctness with task-appropriate metrics
>* Consider error costs and label imbalance



In [ ]:
#@title Python Code - Multilabel Classification

# Demonstrate multilabel classification with indicator targets.
# Each sample can receive several labels.
# The output shows partial and exact correctness.

import numpy as np
import sklearn
from sklearn.datasets import make_multilabel_classification
from sklearn.model_selection import train_test_split

from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score

# Create a small multilabel dataset with three possible labels.
features, labels = make_multilabel_classification(
    n_samples=300, n_features=8, n_classes=3, n_labels=2, random_state=42
)

# Check that the target is a two-dimensional indicator matrix.
if labels.ndim != 2 or labels.shape[1] != 3:
    raise ValueError("Expected a multilabel indicator matrix with three columns.")

# Split features and all label columns together.
X_train, X_test, y_train, y_test = train_test_split(
    features, labels, test_size=0.25, random_state=42
)

# OneVsRestClassifier trains one binary classifier per label column.
base_model = LogisticRegression(max_iter=500, random_state=42)
model = OneVsRestClassifier(base_model)
model.fit(X_train, y_train)

# Predict a set of labels for each test sample.
y_pred = model.predict(X_test)
exact_match = accuracy_score(y_test, y_pred)
micro_f1 = f1_score(y_test, y_pred, average="micro")

# Show a few true and predicted label sets.
label_names = np.array(["sports", "finance", "health"])
true_first = label_names[y_test[0].astype(bool)].tolist()
pred_first = label_names[y_pred[0].astype(bool)].tolist()

print("scikit-learn version:", sklearn.__version__)
print("Target shape:", labels.shape)
print("First true labels:", true_first)
print("First predicted labels:", pred_first)
print("Exact-match accuracy:", round(exact_match, 3))
print("Micro F1 score:", round(micro_f1, 3))



### **2.2. Multioutput Classification**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_19/Lecture_A/image_02_02.jpg?v=1784013789" width="250">



>* Predict several categorical targets per sample
>* Use columns for separate output classes

>* Separate classifiers share the same input features
>* Wrapped estimators handle different output categories

>* Separate outputs differ from shared multilabel sets
>* Evaluate each output and overall performance



In [ ]:
#@title Python Code - Multioutput Classification

# Demonstrate multioutput classification with separate target columns.
# Each output predicts a different categorical question.
# Results show per-output and exact-match accuracy.

import numpy as np
import sklearn
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

from sklearn.multioutput import MultiOutputClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Create shared input features for all output tasks.
features, urgency = make_classification(
    n_samples=600,
    n_features=6,
    n_informative=4,
    n_redundant=0,
    n_classes=3,
    random_state=42,
)

# Build a second categorical output from related features.
department = np.where(features[:, 0] + features[:, 1] > 0, 1, 0)
quality = np.where(features[:, 2] > np.median(features[:, 2]), 1, 0)

# Stack columns into one two-dimensional multioutput target.
targets = np.column_stack((urgency, department, quality))
output_names = ["urgency", "department", "quality"]

# Validate the target shape before fitting the model.
if targets.ndim != 2 or targets.shape[1] != 3:
    raise ValueError("Expected three target columns for multioutput learning.")

# Split features and all target columns together.
X_train, X_test, y_train, y_test = train_test_split(
    features,
    targets,
    test_size=0.25,
    random_state=42,
)

# Wrap one simple classifier for each output column.
base_tree = DecisionTreeClassifier(max_depth=4, random_state=42)
model = MultiOutputClassifier(base_tree)
model.fit(X_train, y_train)

# Predict all output columns for the same test samples.
predictions = model.predict(X_test)
exact_match = np.mean(np.all(y_test == predictions, axis=1))

print(f"scikit-learn version: {sklearn.__version__}")
print(f"Target matrix shape: {targets.shape[0]} rows x {targets.shape[1]} outputs")

# Compare accuracy separately for each output question.
for column_index, output_name in enumerate(output_names):
    score = accuracy_score(y_test[:, column_index], predictions[:, column_index])
    print(f"{output_name} accuracy: {score:.2f}")

print(f"Exact-match accuracy across all outputs: {exact_match:.2f}")



### **2.3. Classifier Chains**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_19/Lecture_A/image_02_03.jpg?v=1784013793" width="250">



>* Model related labels together
>* Use earlier predictions as later inputs

>* Chains train one binary model per label
>* Later predictions use earlier label outputs

>* Label order affects chain performance
>* Errors can propagate, but dependencies help



In [ ]:
#@title Python Code - Classifier Chains

# Demonstrate classifier chains for multilabel classification.
# Earlier label predictions help later label models.
# Compare independent and chained multilabel predictions.

import numpy as np
import sklearn
from sklearn.datasets import make_multilabel_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.multioutput import ClassifierChain
from sklearn.multiclass import OneVsRestClassifier

# Create a small multilabel dataset with related labels.
features, labels = make_multilabel_classification(
    n_samples=600,
    n_features=10,
    n_classes=3,
    n_labels=2,
    random_state=42,
)

# Check that each sample has three binary label columns.
if labels.shape[1] != 3:
    raise ValueError("Expected exactly three label columns.")

# Split features and multilabel targets for fair evaluation.
X_train, X_test, y_train, y_test = train_test_split(
    features,
    labels,
    test_size=0.30,
    random_state=42,
)

# Use the same simple classifier inside both strategies.
base_model = LogisticRegression(max_iter=500, random_state=42)
independent_model = OneVsRestClassifier(base_model)
chain_model = ClassifierChain(base_model, order=[0, 1, 2], random_state=42)

# Fit independent labels and chained labels on training data.
independent_model.fit(X_train, y_train)
chain_model.fit(X_train, y_train)

# Predict all labels for the held-out test examples.
independent_predictions = independent_model.predict(X_test)
chain_predictions = chain_model.predict(X_test)

# Micro F1 summarizes multilabel prediction quality.
independent_f1 = f1_score(y_test, independent_predictions, average="micro")
chain_f1 = f1_score(y_test, chain_predictions, average="micro")

# Show one example where the chain predicts three labels.
example_index = 0
print("scikit-learn version:", sklearn.__version__)
print("Target columns: label_0, label_1, label_2")
print("Independent micro F1:", round(independent_f1, 3))
print("Classifier chain micro F1:", round(chain_f1, 3))
print("True labels for one test sample:", y_test[example_index].tolist())
print("Chain prediction for that sample:", chain_predictions[example_index].astype(int).tolist())



## **3. Chained Regression Outputs**

### **3.1. Multioutput Regression Basics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_19/Lecture_A/image_03_01.jpg?v=1784013795" width="250">



>* Predict several continuous targets per input
>* Outputs form a numeric prediction vector

>* Train separate regressors for each target
>* Works well when outputs are weakly related

>* Simple multioutput regression may miss target relationships
>* Chains use earlier predictions for later outputs



In [ ]:
#@title Python Code - Multioutput Regression Basics

# This script demonstrates basic multioutput regression.
# One model predicts several continuous targets.
# Results show vector predictions and separate errors.

import numpy as np
import sklearn
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

# Create one feature matrix with three numeric targets.
features, targets = make_regression(
    n_samples=300,
    n_features=5,
    n_targets=3,
    noise=12.0,
    random_state=42,
)

# Confirm that each row has multiple target values.
if targets.shape[1] != 3:
    raise ValueError("Expected three target columns.")

# Split the same inputs and all outputs together.
X_train, X_test, y_train, y_test = train_test_split(
    features,
    targets,
    test_size=0.25,
    random_state=42,
)

# The wrapper fits one linear regressor per output column.
model = MultiOutputRegressor(LinearRegression())
model.fit(X_train, y_train)

# Predictions are vectors, not single numbers.
predictions = model.predict(X_test)

# Measure each output separately for clearer interpretation.
errors = mean_absolute_error(
    y_test,
    predictions,
    multioutput="raw_values",
)

print("scikit-learn version:", sklearn.__version__)
print("Target matrix shape:", targets.shape)
print("Prediction vector shape:", predictions.shape)
print("First true target vector:", np.round(y_test[0], 1).tolist())
print("First predicted vector:", np.round(predictions[0], 1).tolist())
print("MAE for outputs 1, 2, 3:", np.round(errors, 1).tolist())



### **3.2. Regressor Chains**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_19/Lecture_A/image_03_02.jpg?v=1784013797" width="250">



>* Predict multiple continuous targets in sequence
>* Use earlier predictions to improve later ones

>* Earlier predictions guide later output estimates
>* Useful dependencies can also spread errors

>* Output order shapes later predictions
>* Validate chains to manage prediction errors



In [ ]:
#@title Python Code - Regressor Chains

# Demonstrate regressor chains for related continuous outputs.
# Compare independent and chained multioutput regression models.
# Show when earlier predictions help later predictions.

import numpy as np
import sklearn
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputRegressor
from sklearn.multioutput import RegressorChain

# Create features and related targets with deterministic randomness.
rng = np.random.default_rng(42)
X = rng.normal(size=(500, 4))

# Later targets depend partly on earlier target values.
y_recovery = 8 + 2.0 * X[:, 0] - 1.5 * X[:, 1] + rng.normal(size=500)
y_pain = 3 + 0.7 * y_recovery + 1.2 * X[:, 2] + rng.normal(size=500)
y_mobility = 20 - 0.9 * y_pain + 0.5 * X[:, 3] + rng.normal(size=500)

# Stack the three related outputs into one target matrix.
y = np.column_stack((y_recovery, y_pain, y_mobility))

# Validate the multioutput shape before modeling.
if y.shape != (500, 3):
    raise ValueError("Expected 500 rows and 3 regression outputs.")

# Split once so both models face the same test examples.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

# Fit independent models that ignore target-to-target dependencies.
independent_model = MultiOutputRegressor(LinearRegression())
independent_model.fit(X_train, y_train)

# Fit a chain that predicts outputs in a meaningful order.
chain_model = RegressorChain(LinearRegression(), order=[0, 1, 2])
chain_model.fit(X_train, y_train)

# Compare average absolute prediction errors on unseen data.
independent_predictions = independent_model.predict(X_test)
chain_predictions = chain_model.predict(X_test)

independent_mae = mean_absolute_error(y_test, independent_predictions)
chain_mae = mean_absolute_error(y_test, chain_predictions)

# Show one example of how the chain predicts all outputs.
example_prediction = chain_model.predict(X_test[:1])[0]
rounded_example = np.round(example_prediction, 2)

print(f"scikit-learn version: {sklearn.__version__}")
print(f"Independent multioutput MAE: {independent_mae:.3f}")
print(f"Regressor chain MAE: {chain_mae:.3f}")
print(f"One chained prediction: {rounded_example}")



### **3.3. Output Dependencies**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_19/Lecture_A/image_03_03.jpg?v=1784013799" width="250">



>* Related outputs can improve predictions
>* Chains use earlier predictions as inputs

>* Prediction order passes information between outputs
>* Earlier predictions reveal useful target relationships

>* Chain order can amplify prediction errors
>* Use real dependencies, not accidental correlations



In [ ]:
#@title Python Code - Output Dependencies

# This example compares independent and chained regression outputs.
# Earlier predictions can help later dependent targets.
# The printed scores show the chain effect.

import numpy as np
import sklearn
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputRegressor
from sklearn.multioutput import RegressorChain

# A fixed generator makes the synthetic data reproducible.
rng = np.random.default_rng(42)

# Two input features describe each made-up machine observation.
sample_count = 300
input_features = rng.normal(size=(sample_count, 2))

# The first target depends directly on the original features.
noise_one = rng.normal(scale=0.4, size=sample_count)
target_temperature = 2.0 * input_features[:, 0] - input_features[:, 1] + noise_one

# The second target also depends on the first target.
noise_two = rng.normal(scale=0.4, size=sample_count)
target_wear = 0.8 * target_temperature + 0.5 * input_features[:, 1] + noise_two

# Multioutput regression stores both targets in one two-column array.
targets = np.column_stack((target_temperature, target_wear))

# This check keeps the lesson clear if shapes are changed later.
if targets.shape != (sample_count, 2):
    raise ValueError("Expected two regression outputs for every sample.")

# Split once so both models face the same test examples.
X_train, X_test, y_train, y_test = train_test_split(
    input_features,
    targets,
    test_size=0.30,
    random_state=42,
)

# The independent model fits one separate regressor per output.
independent_model = MultiOutputRegressor(LinearRegression())
independent_model.fit(X_train, y_train)
independent_predictions = independent_model.predict(X_test)

# The chain predicts temperature first, then uses it for wear.
chain_model = RegressorChain(LinearRegression(), order=[0, 1])
chain_model.fit(X_train, y_train)
chain_predictions = chain_model.predict(X_test)

# R squared is higher when predictions are closer to true values.
independent_wear_r2 = r2_score(y_test[:, 1], independent_predictions[:, 1])
chain_wear_r2 = r2_score(y_test[:, 1], chain_predictions[:, 1])

print(f"scikit-learn version: {sklearn.__version__}")
print("Target order: temperature first, wear second")
print(f"Independent wear R2: {independent_wear_r2:.3f}")
print(f"Chained wear R2: {chain_wear_r2:.3f}")
print("The chain can use earlier output predictions as extra features.")



# <font color="#418FDE" size="6.5" uppercase>**Output Structures**</font>


In this lecture, you learned to:
- Apply multiclass strategies for estimators that need decomposition wrappers. 
- Model multilabel and multioutput classification targets in scikit-learn. 
- Use chained estimators for dependent classification or regression outputs. 

In the next Lecture (Lecture B), we will go over 'Semi Supervised'